# 01 · Load, clean and validate the KTAS triage data

Vitals model, step 1. The data is the Kaggle *Emergency Service – Triage Application* dataset
(1,267 adult visits to two Korean emergency departments, Oct 2016 – Sep 2017).
The target is `KTAS_expert`, the urgency level (1 = most urgent … 5 = least) agreed by three triage experts.

Source paper (cite this in the report): Moon S-H, Shim JL, Park K-S, Park C-S. *Triage accuracy and causes of
mistriage using the Korean Triage and Acuity Scale.* PLoS One 2019;14(9):e0216972.

All cleaning logic lives in `../src/ktas_data.py`, so the app and later notebooks reuse exactly the same steps.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
import ktas_data as kd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 110)

## 1. Load the raw file as text

Everything is read as text first, so nothing is silently converted. The file is semicolon-separated,
Turkish Windows encoding (`cp1254`), and uses decimal commas.

In [2]:
raw = kd.load_raw()
print(raw.shape)
raw.head()

(1267, 24)


,Group,Sex,Age,Patients number per hour,Arrival mode,Injury,Chief_complain,Mental,Pain,NRS_pain,SBP,DBP,HR,RR,BT,Saturation,KTAS_RN,Diagnosis in ED,Disposition,KTAS_expert,Error_group,Length of stay_min,KTAS duration_min,mistriage
0,2,2,71,3,3,2,right ocular pain,1,1,2,160,100,84,18,36.6,100,2,Corneal abrasion,1,4,2,86,"5,00",1
1,1,1,56,12,3,2,right forearm burn,1,1,2,137,75,60,20,36.5,,4,"Burn of hand, firts degree dorsum",1,5,4,64,"3,95",1
2,2,1,68,8,2,2,"arm pain, Lt",1,1,2,130,80,102,20,36.6,98,4,"Fracture of surgical neck of humerus, closed",2,5,4,862,"1,00",1
3,1,2,71,8,1,1,ascites tapping,1,1,3,139,94,88,20,36.5,,4,Alcoholic liver cirrhosis with ascites,1,5,6,108,"9,83",1
4,1,2,58,4,3,1,"distension, abd",1,1,3,91,67,93,18,36.5,,4,Ascites,1,5,8,109,"6,60",1


### What is wrong with the raw file

Problems found by auditing every column:

| Problem | Where | Fix |
|---|---|---|
| Missing vitals written as `??` | SBP, DBP, HR, RR, BT, SpO2, chief complaint | set to missing |
| Missing pain scores written as `#BOŞ!` (Turkish Excel's "#EMPTY!") | NRS_pain | set to missing, then 0 if the patient reported no pain |
| Trailing dot (`36.`) and trailing space (`0 `) | DBP, BT, Error_group, mistriage | stripped before converting to numbers |
| Decimal comma (`5,00`) | KTAS duration | converted to a decimal point |
| 5 visits recorded twice, identical except for a second diagnosis | 10 rows | merged into one row; diagnoses joined with ` \| ` |
| SpO2 = 20% in an alert patient triaged KTAS 3 | 1 row | outside plausible limits, set to missing |

In [3]:
placeholder_counts = pd.DataFrame({p: (raw == p).sum() for p in kd.PLACEHOLDERS})
placeholder_counts[placeholder_counts.sum(axis=1) > 0]

,??,#BOŞ!
Chief_complain,5,0
NRS_pain,0,556
SBP,25,0
DBP,29,0
HR,20,0
RR,22,0
BT,18,0
Saturation,9,0


## 2. Clean

`clean()` returns the cleaned table and a log of every change, so each fix can be reported.

In [4]:
df, log = kd.clean(raw)
print(f"{len(raw)} raw rows -> {len(df)} visits, {df.shape[1]} columns")
log

1267 raw rows -> 1262 visits, 44 columns


,step,column,n_rows,detail
0,rename,all,1267,columns renamed to snake_case
1,placeholder -> missing,chief_complaint,5,'??'
2,placeholder -> missing,nrs_pain,556,'#BOŞ!'
3,placeholder -> missing,sbp,25,'??'
4,placeholder -> missing,dbp,29,'??'
5,placeholder -> missing,hr,20,'??'
6,placeholder -> missing,rr,22,'??'
7,placeholder -> missing,temp_c,18,'??'
8,placeholder -> missing,spo2,9,'??'
9,trailing dot,dbp,1,e.g. '36.' read as 36.0


## 3. Validate

Every check should pass. A check marked `warning` is a known gap that is kept on purpose.

In [5]:
report = kd.validate(df)
errors = report[(report.severity == "error") & ~report.passed]
assert errors.empty, errors
report

,check,passed,n_failing,severity,detail
0,one row per visit (no repeated triage records),True,0,error,
1,visit_id unique,True,0,error,
2,all expected columns present,True,0,error,
3,ktas_expert in 1-5 and never missing,True,0,error,
4,ktas_nurse in 1-5 and never missing,True,0,error,
5,ed_type codes valid,True,0,error,"allowed [1, 2]"
6,sex codes valid,True,0,error,"allowed [1, 2]"
7,injury codes valid,True,0,error,"allowed [1, 2]"
8,pain codes valid,True,0,error,"allowed [0, 1]"
9,mental codes valid,True,0,error,"allowed [1, 2, 3, 4]"


## 4. Missing data is not random

About half of KTAS 1 patients have no vitals recorded, compared with under 1% for KTAS 3–4.
SpO2 is missing for over half of all patients.

In [6]:
kd.missingness(df)

,missing,missing_%,KTAS 1 %,KTAS 2 %,KTAS 3 %,KTAS 4 %,KTAS 5 %
sbp,25,2.0,56.0,2.7,0.2,0.2,4.0
dbp,29,2.3,64.0,3.6,0.2,0.2,4.0
hr,20,1.6,48.0,1.8,0.0,0.2,4.0
rr,22,1.7,64.0,0.9,0.0,0.2,4.0
temp_c,17,1.3,44.0,0.9,0.0,0.2,4.0
spo2,693,54.9,68.0,33.2,55.7,60.4,76.0
nrs_pain,3,0.2,NaN,NaN,NaN,NaN,NaN


Most KTAS 1 patients with no vitals were in cardiac arrest, unresponsive or post-CPR, too unwell to measure.

**Consequence for the vitals model:** NEWS2 normally scores a missing parameter as 0, which would make a cardiac-arrest
patient look *low risk*. The vitals model must report missing values explicitly, and treat
"unresponsive with vitals missing" as danger, never as normal.

In [7]:
cols = ["visit_id", "chief_complaint", "mental_label", "sbp", "hr", "rr", "spo2",
        "n_vitals_missing", "diagnosis", "disposition_label"]
df.loc[(df.ktas_expert == 1) & (df.n_vitals_missing >= 4), cols]

,visit_id,chief_complaint,mental_label,sbp,hr,rr,spo2,n_vitals_missing,diagnosis,disposition_label
121,122,dyspnea,alert,NaN,NaN,NaN,NaN,6,Dyspnea,discharge
149,150,mental change,pain,NaN,NaN,NaN,NaN,6,End stage kidney disease on dialysis,ward
154,155,Motor weakness,unresponsive,NaN,NaN,NaN,NaN,6,Dysarthria,ward
160,161,melena,pain,NaN,NaN,24.0,NaN,4,Hematochezia,icu
702,703,abd pain,alert,NaN,NaN,NaN,NaN,6,Contusion of abdomen wall,death
817,818,mental change,unresponsive,NaN,NaN,NaN,NaN,6,Cardiac arrest,transfer
884,885,seizure,unresponsive,NaN,NaN,NaN,NaN,6,Seizure,discharge_code4
1006,1007,post-CPR state,unresponsive,NaN,NaN,NaN,NaN,6,"Cardiac arrest, unspecified",death
1035,1036,arrest,unresponsive,NaN,NaN,NaN,NaN,6,Cardiac arrest,death
1074,1075,altered mental change,unresponsive,NaN,NaN,NaN,NaN,5,Intracerebral hemorrhage,icu


## 5. Summary of the cleaned data

In [8]:
df[kd.VITALS + ["nrs_pain", "age"]].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
sbp,1237.0,133.7,27.2,50.0,114.0,130.0,150.0,275.0
dbp,1233.0,79.8,15.1,31.0,70.0,80.0,90.0,160.0
hr,1242.0,84.0,16.6,32.0,72.0,82.0,96.0,148.0
rr,1240.0,19.5,2.0,14.0,18.0,20.0,20.0,30.0
temp_c,1245.0,36.6,0.5,35.0,36.2,36.5,36.8,41.0
spo2,569.0,97.2,2.9,68.0,97.0,98.0,98.0,100.0
nrs_pain,1259.0,2.3,2.3,0.0,0.0,3.0,4.0,10.0
age,1262.0,54.4,19.8,16.0,37.0,57.0,71.0,96.0


In [9]:
counts = df.ktas_expert.value_counts().sort_index()
pd.DataFrame({"visits": counts, "%": (counts / len(df) * 100).round(1)}).rename_axis("KTAS_expert")

,visits,%
KTAS_expert,,
1,25,2.0
2,220,17.4
3,485,38.4
4,457,36.2
5,75,5.9


KTAS 1 has only 25 visits, so any train/test split must be **stratified** by `ktas_expert`.

Nurse vs expert triage (rows = nurse, columns = expert). Above the diagonal = nurse under-triaged (rated less urgent than the experts).

In [10]:
pd.crosstab(df.ktas_nurse, df.ktas_expert, margins=True)

ktas_expert,1,2,3,4,5,All
ktas_nurse,,,,,,
1,15,3,0,0,0,18
2,10,183,16,4,0,213
3,0,27,398,20,0,445
4,0,6,63,418,12,499
5,0,1,8,15,63,87
All,25,220,485,457,75,1262


## 6. Save

In [11]:
out = kd.save(df, log)
print("Saved to", out.resolve())
print(sorted(p.name for p in out.iterdir()))

Saved to C:\Users\Claeb\anaconda3\BN4601a\group_project\vitals\data\processed
['cleaning_log.csv', 'ktas_clean.csv', 'news2_scores.csv', 'splits.csv', 'test_predictions.csv']


## Which columns may be used

- **Model inputs** (known at the triage desk): `kd.TRIAGE_FEATURES`
- **Target:** `ktas_expert`
- **Never use as inputs** (known only after triage, or another version of the answer): `kd.LEAKAGE_COLS`.
  `ktas_nurse` is kept only as the human benchmark.

**Next notebook:** stratified train/validation/test split, then the NEWS2 calculator, evaluated against `ktas_expert`.

In [12]:
print("Inputs :", kd.TRIAGE_FEATURES)
print("Target :", kd.TARGET)
print("Leakage:", kd.LEAKAGE_COLS)

Inputs : ['age', 'is_male', 'arrival_mode', 'injured', 'chief_complaint', 'mental', 'has_pain', 'nrs_pain', 'sbp', 'dbp', 'hr', 'rr', 'temp_c', 'spo2', 'ed_type', 'patients_per_hour']
Target : ktas_expert
Leakage: ['ktas_nurse', 'diagnosis', 'disposition', 'error_group', 'los_min', 'ktas_duration_min', 'mistriage']
